### memanggil data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# ganti path sesuai lokasi file kamu di Drive
DATA_PATH = "/content/drive/MyDrive/dataYoutube_disabilitas_2023_2026.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(14505, 10)


,video_id,video_title,comment_id,is_reply,parent_id,author,text,likeCount,publishedAt,is_relevant_keyword
0,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa P...,Ugy__Xbktb9UbUgv5M94AaABAg,False,NaN,@ourthings5977,dongo,0,2023-12-03T16:21:48Z,False
1,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa P...,Ugxmuu_b9zZ0DU_Ptad4AaABAg,False,NaN,@Hellebæk-Ålsgårde_009,Geram gue liat ibu mentri menolak kodrat Allah.,0,2023-08-25T01:00:25Z,False
2,sfeRpL4t6yY,Menteri Risma Dikritik Masyarakat Usai Paksa P...,UgwGj1x5QwlQQHqiYO54AaABAg,False,NaN,@elmiutami1515,"Maklumi aja,,namanya juga bukan guru disabilit...",0,2023-02-16T05:41:07Z,True
3,HqYHDaPKWn8,Supir tunarungu mendapat penumpang spesial,UgwyeViLdPYy4CNt0n54AaABAg,False,NaN,@NurainiAini-us5dr,😢😢😅❤😂,0,2026-02-01T06:14:58Z,False
4,HqYHDaPKWn8,Supir tunarungu mendapat penumpang spesial,Ugz7JNkwYLHT-bsvRVp4AaABAg,False,NaN,@nikikulo-x5l,❤❤❤ ..,0,2025-10-21T12:08:59Z,False


### Persiapan Data

In [ ]:
print(f"Jumlah data awal: {len(df)} baris")

Jumlah data awal: 14505 baris


In [ ]:
import re

def has_letter(text: str) -> bool:
    """Cek apakah teks mengandung minimal satu huruf (bukan cuma emoji/simbol)."""
    return bool(re.search(r"[a-zA-Z]", text))

n_awal = len(df)

# 1. Hapus missing value / teks kosong
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""]

# 2. Hapus duplikat teks persis
df = df.drop_duplicates(subset=["text"], keep="first")

# 3. Hapus komentar < 3 karakter
df = df[df["text"].str.len() >= 3]

# 4. Pisahkan dulu komentar yang murni emoji/simbol (tanpa huruf) sebelum dibuang,
#    supaya masih bisa dipakai untuk analisis tambahan nanti
df_emoji_only = df[~df["text"].apply(has_letter)].copy()
EMOJI_ONLY_PATH = "/content/drive/MyDrive/data_emoji_only.csv"
df_emoji_only.to_csv(EMOJI_ONLY_PATH, index=False, encoding="utf-8-sig")
print(f"Komentar emoji-only dipisah & disimpan: {len(df_emoji_only)}")

# baru hapus dari df utama
df = df[df["text"].apply(has_letter)]
df = df.reset_index(drop=True)

print(f"Data awal: {n_awal} -> Data bersih (df utama): {len(df)}")

Komentar emoji-only dipisah & disimpan: 160
Data awal: 14505 -> Data bersih (df utama): 13656


In [ ]:
# simpan data bersih ke Drive
CLEAN_PATH = "/content/drive/MyDrive/data_clean_tahap1.csv"
df.to_csv(CLEAN_PATH, index=False, encoding="utf-8-sig")
print(f"Tersimpan di: {CLEAN_PATH}")

Tersimpan di: /content/drive/MyDrive/data_clean_tahap1.csv


### Ekstraksi emoji

In [ ]:
!pip install emoji
import emoji

def extract_emojis(text: str) -> list:
    """Ambil semua karakter emoji dari teks."""
    return [c for c in text if c in emoji.EMOJI_DATA]

df["emoji_list"] = df["text"].apply(extract_emojis)
df["emoji_count"] = df["emoji_list"].apply(len)
df["emoji_str"] = df["emoji_list"].apply(lambda x: " ".join(x))  # versi string, aman disimpan ke CSV

print(f"Komentar yang mengandung emoji: {(df['emoji_count'] > 0).sum()} dari {len(df)}")
df[df["emoji_count"] > 0][["text", "emoji_str", "emoji_count"]].head(10)

Komentar yang mengandung emoji: 4836 dari 13656


,text,emoji_str,emoji_count
3,Teharu🙏 penupang,🙏,1
4,"Bahasa terindah yg ingin ku pelajari, ya baha...",❤,1
7,Auto.jdi bestie...😂😂😂,😂 😂 😂,3
15,Kamu luar biasa! 🍁 𝐓𝐈𝐌𝐎𝟒𝐃 🍁 selalu mendukung l...,🍁 🍁,2
17,Keren parah 𝐓𝐈𝐌𝐎𝟒𝐃! Pengen nonton ulang. Salut! 👏,👏,1
18,"Kontennya bikin betah, 👵𝐓𝐈𝐌𝐎𝟒𝐃👵 juga bikin bet...",👵 👵 🌄,3
20,Streamer favorit ❤ dan selalu menunggu update ...,❤ 😁 😁 💪 💪,5
23,Terpopuler ⭐𝐓𝐈𝐌𝐎𝟒𝐃⭐ Member baru pasti jepee❤,⭐ ⭐ ❤,3
24,"Mantep ⭐𝐓𝐈𝐌𝐎𝟒𝐃⭐ udah jadi kebiasaan, Caerr ter...",⭐ ⭐,2
25,Bangun tidur langsung buka ✌𝐓𝐈𝐌𝐎𝟒𝐃✌.,✌ ✌,2


### Hapus komentar spam

In [ ]:
import re

# pola umum: huruf unicode bold/mathematical alphanumeric, sering dipakai bot spam
def has_unicode_bold(text: str) -> bool:
    return bool(re.search(r'[\U0001D400-\U0001D7FF]', text))

spam_keywords = ["timo4d", "slot", "gacor", "maxwin", "wd", "deposit", "jp", "member baru"]

df["is_suspected_spam"] = df["text"].apply(has_unicode_bold) | \
    df["text"].str.lower().str.contains("|".join(spam_keywords), regex=True)

print(f"Terindikasi spam: {df['is_suspected_spam'].sum()} dari {len(df)}")
df[df["is_suspected_spam"]][["text"]].head(15)

Terindikasi spam: 43 dari 13656


,text
14,0:02 Wah keren 𝐓𝐈𝐌𝐎𝟒𝐃 terbaik menang mudah
15,Kamu luar biasa! 🍁 𝐓𝐈𝐌𝐎𝟒𝐃 🍁 selalu mendukung l...
16,"Tetap semangat ya, kami selalu dukung! 𝗧𝗶𝗺𝗼𝟰𝗗"
17,Keren parah 𝐓𝐈𝐌𝐎𝟒𝐃! Pengen nonton ulang. Salut! 👏
18,"Kontennya bikin betah, 👵𝐓𝐈𝐌𝐎𝟒𝐃👵 juga bikin bet..."
19,Mantap 𝐓𝐈𝐌𝐎𝟒𝐃 suka pecah scatter pantes rame
20,Streamer favorit ❤ dan selalu menunggu update ...
21,Mantap! 𝗧𝗶𝗺𝗼𝟰𝗗
22,Konten seperti ini yang dibutuhkan di YouTube....
23,Terpopuler ⭐𝐓𝐈𝐌𝐎𝟒𝐃⭐ Member baru pasti jepee❤


In [ ]:
n_before_spam = len(df)

# simpan dulu contoh yang kebuang, buat dokumentasi/lampiran skripsi kalau dibutuhkan
df_spam = df[df["is_suspected_spam"]].copy()
SPAM_PATH = "/content/drive/MyDrive/data_spam_terdeteksi.csv"
df_spam.to_csv(SPAM_PATH, index=False, encoding="utf-8-sig")

# buang dari df utama
df = df[~df["is_suspected_spam"]].copy()
df = df.drop(columns=["is_suspected_spam"])
df = df.reset_index(drop=True)

print(f"Data sebelum buang spam: {n_before_spam}")
print(f"Data setelah buang spam: {len(df)} (terhapus {n_before_spam - len(df)})")

# update file bersih di Drive
CLEAN_PATH = "/content/drive/MyDrive/data_clean_tahap1.csv"
df.to_csv(CLEAN_PATH, index=False, encoding="utf-8-sig")
print(f"data_clean_tahap1.csv diperbarui: {len(df)} baris")

Data sebelum buang spam: 13656
Data setelah buang spam: 13613 (terhapus 43)
data_clean_tahap1.csv diperbarui: 13613 baris


In [ ]:
# update file bersih di Drive
CLEAN_PATH = "/content/drive/MyDrive/data_clean_tahap1.csv"
df.to_csv(CLEAN_PATH, index=False, encoding="utf-8-sig")
print(f"data_clean_tahap1.csv diperbarui: {len(df)} baris")

data_clean_tahap1.csv diperbarui: 13613 baris
